# 02 — Fine-tune SegFormer on FoodSeg103 (GPU job 1)

Trains the on-device food segmenter. Every public FoodSeg103 checkpoint measures at mIoU ≤ 0.05 (docs/MODELS.md), so we train our own.

- **Runtime:** H100 — B0 ~2–3 h, B1 ~4–6 h (60 epochs, batch 32). Select the GPU runtime before running.
- **Targets:** mIoU ≥ 0.25 (mit-b0) / ≥ 0.32 (mit-b1)
- **Everything persists to Drive** — the FoodSeg103 dataset cache, the pretrained backbone, and per-epoch checkpoints — so a disconnect loses one epoch at most; rerun and `Trainer` resumes from the last checkpoint. Notebook 01 is NOT required for this one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

# All Hugging Face + torch caches live on Drive, so datasets and pretrained
# weights download once and survive VM recycling.
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf-cache'
os.environ['TORCH_HOME'] = f'{DRIVE_ROOT}/torch-cache'

MODEL = 'nvidia/mit-b0'   # or 'nvidia/mit-b1'
EPOCHS = 60
BATCH = 32
OUT = f"{DRIVE_ROOT}/checkpoints/segformer-{MODEL.split('/')[-1]}-food"

!nvidia-smi | head -12

In [ ]:
!git clone --depth 1 https://github.com/Hakeem-Hannoon/physics-powered-portion-estimation.git /content/ppe 2>/dev/null || (cd /content/ppe && git pull)
%pip -q install transformers datasets evaluate accelerate timm

In [ ]:
%cd /content/ppe
!python model/train/segformer_foodseg103.py \
  --model "{MODEL}" --epochs {EPOCHS} --batch-size {BATCH} \
  --output "{OUT}"

In [ ]:
# Result row for the README 'Testing set & results' table.
import glob, json
states = sorted(glob.glob(f'{OUT}/checkpoint-*/trainer_state.json'))
best = json.load(open(states[-1])).get('best_metric')
print(f'best mIoU: {best}')
print(f'| FoodSeg103 val | mIoU | public ≤ 0.05; B0 ≈ 0.25, B1 ≈ 0.32 | **{best:.3f}** ({MODEL}) |')